# Onchain Transaction Data Processing

### **Importing polars**

to process the dataset with memory efficiently as the dataset is huge

In [16]:
import polars as pl
import os

In [3]:
daily = pl.read_parquet('../data/raw/chain/daily_filtered.parquet')
targets = pl.read_parquet('../data/raw/chain/targets_global.parquet')

### **Explore the Datasets**

In [4]:
print("\ndaily shape:", daily.shape)
print("\ndaily column types:", daily.schema)
print("\ndaily date range:", daily['day'].min(), "-", daily['day'].max())
print("\ndaily address count:", daily['address'].n_unique())


daily shape: (42946110, 31)

daily column types: Schema({'node_id': UInt64, 'address': String, 'day': Date, 'week': String, 'month': String, 'normal_sent_cnt': Int64, 'normal_recv_cnt': Int64, 'normal_total_cnt': Int64, 'normal_failed_cnt': Int64, 'normal_to_contract_cnt': Int64, 'normal_to_eoa_cnt': Int64, 'gas_used_sum': Int64, 'erc20_sent_cnt': Int64, 'erc20_recv_cnt': Int64, 'erc20_total_cnt': Int64, 'erc20_unique_tokens_sent': Int64, 'erc20_unique_tokens_recv': Int64, 'internal_out_cnt': Int64, 'internal_in_cnt': Int64, 'uniq_peers_cnt': Int64, 'uniq_contract_peers_cnt': Int64, 'uniq_eoa_peers_cnt': Int64, 'sessions_cnt': Int64, 'active_span_min': Int64, 'burst_max_tx_5m': Int64, 'eth_sent_sum': Decimal(precision=38, scale=9), 'eth_recv_sum': Decimal(precision=38, scale=9), 'eth_net_flow': Decimal(precision=38, scale=9), 'tx_fee_eth_sum': Decimal(precision=38, scale=9), 'internal_out_value_eth_sum': Decimal(precision=38, scale=9), 'internal_in_value_eth_sum': Decimal(precision=38

In [5]:
print("\ntargets shape:", targets.shape)
print("\ntargets schema:", targets.schema)
print("\ntarget address count:", targets['address'].n_unique())
print("\nclass balance:", targets.group_by('is_scam').len())
print("\ncontract count", targets.group_by('is_contract').len())


targets shape: (114739, 4)

targets schema: Schema({'node_id': UInt64, 'is_scam': Int8, 'is_contract': Int8, 'address': String})

target address count: 114739

class balance: shape: (2, 2)
┌─────────┬───────┐
│ is_scam ┆ len   │
│ ---     ┆ ---   │
│ i8      ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 60086 │
│ 1       ┆ 54653 │
└─────────┴───────┘

contract count shape: (2, 2)
┌─────────────┬───────┐
│ is_contract ┆ len   │
│ ---         ┆ ---   │
│ i8          ┆ u32   │
╞═════════════╪═══════╡
│ 1           ┆ 69698 │
│ 0           ┆ 45041 │
└─────────────┴───────┘


In [6]:
# check overlap
overlap = daily.select('address').unique().join(targets.select('address').unique(), on='address', how='inner')

print("\naddresses not in daily:", targets['address'].n_unique() - overlap['address'].n_unique())


addresses not in daily: 29592


### **Filter EOAs and Join with Daily Data**

In [7]:
eoa = targets.filter(pl.col("is_contract") == 0)
print("\nEOA count:", eoa.height)

# overlap of EOAs is_contract=0 with daily data
eoa_overlap = daily.select("address").unique().join(eoa.select("address").unique(), on="address", how="inner")

print("\noverlap EOA addresses with daily data:", eoa_overlap["address"].n_unique())


EOA count: 45041

overlap EOA addresses with daily data: 42942


In [8]:
eoa_targets = targets.filter(pl.col("is_contract") == 0)

eoa_address_set = set(
    eoa_targets
    .select("address")
    .unique()
    .to_series()
    .to_list()
)

eoa_daily = daily.filter(pl.col("address").is_in(eoa_address_set))

print("\neoa filtered class balance:", eoa_targets.group_by('is_scam').len())


eoa filtered class balance: shape: (2, 2)
┌─────────┬───────┐
│ is_scam ┆ len   │
│ ---     ┆ ---   │
│ i8      ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 30059 │
│ 1       ┆ 14982 │
└─────────┴───────┘


### **Convert Decimal Columns to Float**

In [9]:
# convert decimal columns to Float64
decimal_cols = ['eth_sent_sum', 'eth_recv_sum', 'eth_net_flow']

eoa_daily = eoa_daily.with_columns([pl.col(col).cast(pl.Float64) for col in decimal_cols])

### **Check for feature inconsistencies**

In [10]:
# columns that indicate activity
check_cols = [
    "normal_sent_cnt",
    "normal_recv_cnt",
    "eth_sent_sum",
    "eth_recv_sum",
    "uniq_peers_cnt",
    "sessions_cnt",
    "active_span_min",
    "burst_max_tx_5m"
]

# inconsistency rule - if normal_total_cnt == 0, then no activity column should be > 0
inconsistent = ((pl.col("normal_total_cnt") == 0) & pl.any_horizontal([pl.col(c) > 0 for c in check_cols]))

# counts
original_rows = eoa_daily.height
num_inconsistent = eoa_daily.select(inconsistent.sum()).item()

zero_activity_rows = eoa_daily.select(((pl.col("normal_total_cnt") == 0) & (~inconsistent)).sum()).item()

# keep only active days
daily_cleaned = eoa_daily.filter(pl.col("normal_total_cnt") > 0)

removed_rows = original_rows - daily_cleaned.height
remaining_rows = daily_cleaned.height

print(f"original rows: {original_rows}")
print(f"inconsistent rows: {num_inconsistent}")
print(f"0 active day rows (without inconsistent): {zero_activity_rows}")
print(f"removed rows: {removed_rows}")
print(f"remaining rows: {remaining_rows}")


original rows: 27682041
inconsistent rows: 18540
0 active day rows (without inconsistent): 26384625
removed rows: 26403165
remaining rows: 1278876


### Calculate days between transactions

In [11]:
daily_cleaned = daily_cleaned.sort(["address", "day"])
daily_cleaned = daily_cleaned.with_columns(
    pl.col("day")
    .diff()
    .over("address")
    .dt.total_days()
    .fill_null(0)
    .alias("days_since_last_activity")
)

In [12]:
keep_cols = [
    "address",
    "day",
    "normal_sent_cnt",
    "normal_recv_cnt",
    "normal_total_cnt",
    "eth_sent_sum",
    "eth_recv_sum",
    "eth_net_flow",
    "uniq_peers_cnt",
    "sessions_cnt",
    "active_span_min",
    "burst_max_tx_5m",
    "days_since_last_activity"
]

daily_cleaned = daily_cleaned.select(keep_cols)
targets_cleaned = eoa_targets.join(daily_cleaned.select("address").unique(), on="address", how="inner")

In [13]:
print("\ndaily shape:", daily_cleaned.shape)
print("\ndaily column types:", daily_cleaned.schema)
print("\ndaily date range:", daily_cleaned['day'].min(), "-", daily_cleaned['day'].max())
print("\ndaily address count:", daily_cleaned['address'].n_unique())


daily shape: (1278876, 13)

daily column types: Schema({'address': String, 'day': Date, 'normal_sent_cnt': Int64, 'normal_recv_cnt': Int64, 'normal_total_cnt': Int64, 'eth_sent_sum': Float64, 'eth_recv_sum': Float64, 'eth_net_flow': Float64, 'uniq_peers_cnt': Int64, 'sessions_cnt': Int64, 'active_span_min': Int64, 'burst_max_tx_5m': Int64, 'days_since_last_activity': Int64})

daily date range: 2015-08-07 - 2025-08-17

daily address count: 42476


In [14]:
print("\ntargets shape:", targets_cleaned.shape)
print("\ntargets schema:", targets_cleaned.schema)
print("\ntarget address count:", targets_cleaned['address'].n_unique())
print("\nclass balance:", targets_cleaned.group_by('is_scam').len())


targets shape: (42476, 4)

targets schema: Schema({'node_id': UInt64, 'is_scam': Int8, 'is_contract': Int8, 'address': String})

target address count: 42476

class balance: shape: (2, 2)
┌─────────┬───────┐
│ is_scam ┆ len   │
│ ---     ┆ ---   │
│ i8      ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 30042 │
│ 1       ┆ 12434 │
└─────────┴───────┘


In [19]:
output_dir = '../data/processed/chain'
os.makedirs(output_dir, exist_ok=True)

parquet_path = os.path.join(output_dir, 'daily_cleaned.parquet')
daily_cleaned.write_parquet(parquet_path)

parquet_path = os.path.join(output_dir, 'targets_global.parquet')
targets_cleaned.write_parquet(parquet_path)

print("Saved")

Saved
